# Deep RecSys Course
## Домашнее задание 2

### ФИО: Хамидуллин Амир Айдарович

В этом домашнем задании вы реализуете различные лосс-функции, которые используются для обучения двухбашенных моделей для стадии отбора кандидатов.

### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

В этом задании нас будет интересовать только файл с взаимодействиями, `interactions.parquet`

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы скачиваем датасет,поэтому самостоятельно его можно не качать.

### Разбалловка
1) Создание датасета и коллатора для обучения - 1 балл
2) Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) - 1 балл
3) Цикл обучения - 1 балл
4) Softmax loss - 1 балл
5) BCE loss - 1 балл
5) BPR loss - 1 балл
6) Sampled softmax, uniform negatives - 1 балл
7) Sampled softmax, in-batch negatives - 1 балл
8) Sampled softmax, in-batch negatives + logq correction - 1 балл
9) Ответы на вопросы в конце ноутбука - 1 балл

In [2]:
!wget -O tests.py "https://raw.githubusercontent.com/KhrylchenkoKirill/DeepRecSys/main/homeworks/hw2/tests.py"

import importlib
import tests
importlib.reload(tests)


--2026-03-26 16:19:48--  https://raw.githubusercontent.com/KhrylchenkoKirill/DeepRecSys/main/homeworks/hw2/tests.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9590 (9.4K) [text/plain]
Saving to: ‘tests.py’

tests.py            100%[===================>]   9.37K  --.-KB/s    in 0s      

2026-03-26 16:19:48 (99.7 MB/s) - ‘tests.py’ saved [9590/9590]



<module 'tests' from '/content/tests.py'>

In [3]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=6008d36f-dda5-499c-bba9-7331db2d92a2
To: /content/dataset.zip
100% 356M/356M [00:02<00:00, 159MB/s]


In [7]:
from collections import defaultdict
import copy
import gc
import os
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests

# 0. Подготовка данных и метрики (код для пропусков возьмите с вашей ДЗ 1)

Обработка данных

In [8]:
# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)


data = data.join(embeddings, on="item_id", how="semi")

temr_df = data['item_id'].value_counts()
valid_df = temr_df.filter(pl.col('count') >= CORE_MIN_INTERACTIONS_PER_ITEM)
data = data.join(valid_df,on ="item_id", how = "semi" )
data = data.join(artists, on = "item_id", how = "left")

# Ремаппинг item_id в непрерывный диапазон [0, N-1].  РАЗОБРАТЬСЯ=================================================
unique_items = data["item_id"].unique().sort()
item_map = pl.DataFrame({
    "item_id": unique_items,
    "item_idx": pl.arange(0, len(unique_items), eager=True),
})
data = data.join(item_map, on="item_id").drop("item_id").rename({"item_idx": "item_id"})
print(f"catalog_size: {data['item_id'].n_unique()}")
print(f"max item_id: {data['item_id'].max()}")  # должно быть = n_unique - 1



split_point = data["timestamp"].max() - TEST_INTERVAL_SECONDS
train_df = data.filter(pl.col("timestamp") < split_point)
test_df = data.filter(pl.col("timestamp") >= split_point)

user_relat = train_df['uid'].value_counts()
test_df = test_df.join(user_relat, on="uid", how = "semi")

user_item = test_df.group_by("uid").agg(pl.col('item_id'))
test_targets = dict(user_item.iter_rows())



catalog_size: 157357
max item_id: 157356


Метрики

In [9]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:

    hits = [1 if conditate in set(targets) else 0 for conditate in candidates[:topk]]
    hitrate = int(sum(hits) > 0) # 0 or 1

    recall = sum(hits)/min(len(targets),topk)

    dcg = sum(hits[k-1]/np.log2(k+1) for k in range(1,topk+1))
    idcg = sum(1/np.log2(k+1) for k in range(1, min(topk, len(targets))+ 1 ))
    n_DCG = dcg/idcg if idcg else 0.0

    return {
        "hitrate": hitrate,
        "recall": recall,
        "ndcg": n_DCG,
    }


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    total = {"hitrate": 0.0, "recall": 0.0, "ndcg": 0.0}

    count_users = 0
    for user_id, target in targets.items():
        canditate = candidates[user_id]

        metrics_user = get_metrics(target,canditate, topk)
        for key in total:
            total[key] += metrics_user[key]

        count_users += 1

    total = {key:val/count_users for key,val in total.items()}

    all_canditates_unique = len(set([item for cand in candidates.values() for item in cand[:topk]]))
    # catalog_size_uniq = train.n_unique()
    coverage = all_canditates_unique/catalog_size

    total['coverage'] = coverage

    return total


# 1. Создание датасета и коллатора для обучения (1 балл)

В этом задании вам нужно реализовать несколько вспомогательных функций для работы с пользовательскими историями переменной длины. Эти функции будут использоваться дальше при построении семплов, батчей и обучении моделей, поэтому важно сразу договориться о формате представления последовательностей.

#### Формат данных: flatten-представление истории

Вместо того чтобы хранить историю каждого пользователя как отдельный массив (и затем делать padding до общей длины), мы будем хранить всю историю батча в одном “плоском” тензоре — в формате flatten:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

То есть в одном тензоре подряд записаны взаимодействия пользователя 1, затем пользователя 2, и так далее.

Чтобы при этом не потерять границы между пользователями, отдельно хранится тензор `length`, где указано, сколько элементов истории относится к каждому пользователю в батче:

`length = [L1, L2, ..., LB]`

где `B` — размер батча, а `Li` — длина истории `i`-го пользователя.


В таком формате будет храниться именно пользовательская история(последовательность взаимодействий), а ваша задача — реализовать функции, которые позволяют:
- восстанавливать границы последовательностей по `length`,
- превращать flatten-представление в padded-формат + mask,
- готовить батч к подаче в модель.

### Функция `create_masted_tensor`

Напишите функцию `create_masked_tensor`, которая по `flatten` представлению батча последовательностей и их длинам формирует `padded` тензор и булеву маску позиций элементов.

In [7]:
def create_masked_tensor(data: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
  """
  Converts a batch of flattened variable-length sequences into a padded tensor and mask.
  Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

  Parameters
  ----------
  data : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
  lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

  Returns
  -------
  Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

  Examples
  --------
  >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
  >>> lengths = torch.tensor([2, 3, 1])
  >>> padded, mask = create_masked_tensor(data, lengths)
  >>> padded
  tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
  >>> mask
  tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
  """
  full_size = (len(lengths), max(lengths)) + data.shape[1:] # заъватываем embed_size если он есть
  padded_tensor = torch.zeros(full_size,dtype = data.dtype)
  columns = torch.arange(max(lengths)) #(cnt,)
  lengths_col = lengths.unsqueeze(1) # (n, 1)

  mask = columns < lengths_col

  padded_tensor[mask] = data
  return (padded_tensor, mask)



In [8]:
# data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
# lengths = torch.tensor([2, 3, 1])
# x = torch.split(data, lengths.tolist())
# # print(x)
# # print(torch.nn.utils.rnn.pad_sequence(x, batch_first=True))

# mask = torch.arange(0, max(lengths))
# print(mask)
# lengths_column = torch.unsqueeze(lengths, 1)
# # lengths_column_2 = lengths.unsqueeze(1)
# print(lengths_column_2)
# print(lengths_column)
# mask_columns = mask.unsqueeze(0)
# print(mask_columns)

# mask = mask < lengths_column
# print(mask)


In [9]:
# import torch

# def pad_user_history(clicks: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
#     matrix = torch.zeros((len(lengths), max(lengths)),dtype = clicks.dtype)
#     columns = torch.arange(max(lengths)) # (n,)
#     print(columns.shape)
#     mask = lengths.unsqueeze(1) > columns
#     # print(mask)


#     matrix[mask] = clicks

#     return matrix


# clicks = torch.tensor([101, 102, 201, 202, 203, 301])

# lengths = torch.tensor([2, 3, 1])
# print(lengths.shape)
# result = pad_user_history(clicks, lengths)
# print(result)

# s = torch.tensor([1,2,3,4])
# print(s)
# s = s.unsqueeze(-2)
# print(s)
# print(s.shape)

In [10]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Класс `YambdaDataset`

Реализуйте класс `YambdaDataset`, который работает с пользовательскими историями взаимодействий и подготавливает семплы для последующего обучения моделей.

Датасет должен поддерживать два режима работы, задаваемые флагом `is_train`, а также обрезку истории до последних `max_seq_len` элементов.

В train-mode мы превращаем одну пользовательскую историю длины `T` в `T-1` обучающих семплов. То есть для пользователя с историей `[i1, i2, ..., iT]` мы создаём семплы с префиксами:

- `history[:1] -> label = i2`
- `history[:2] -> label = i3`
- ...
- `history[:T-1] -> label = iT`

Если реализовать это “в лоб” и в `__init__` материализовать все такие семплы, то мы получим сильное дублирование данных: один и тот же айтем `i1` будет повторяться почти во всех семплах, `i2` — во всех, кроме первого, и т.д.

Поэтому в `__init__` мы храним только индексы/указатели, а сам префикс от `history` и обрезку строим на лету в `__getitem__`.

#### Входные данные

- `histories: Dict[uid, List[int]]` — временно упорядоченные истории пользовательских взаимодействий.
- `labels: Dict[uid, List[int]]` — целевые айтемы пользователя для оценки (в нашем случае последняя неделя).
- `is_train: bool` — режим работы датасета.
- `max_seq_len: int` — максимальная длина возвращаемой истории (по умолчанию `100`).

#### Режим 1: Train mode (`is_train=True`)

В режиме обучения датасет должен подготовить семплы на пользователя в постановке next-item prediction.

Если история пользователя: `[i1, i2, ..., iT]`, то нужно создать `T - 1` семплов. Для каждого `t` от `1` до `T-1` (позиция следующего айтема):

- `history` = префикс `history[:t]`, обрезанный до последних `max_seq_len` элементов
- `label` = следующий айтем `history[t]`

Формат train-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Режим 2: Inference mode (`is_train=False`)

В режиме оценки датасет должен возвращать ровно один семпл на пользователя.
Пользователь попадает в датасет только если для него есть таргеты в `labels`.
Содержимое семпла:
- `history` = история пользователя, обрезанная до последних `max_seq_len` элементов.

Формат eval-семпла:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [18]:
class YambdaDataset(Dataset):
  """
  PyTorch Dataset for user interaction histories with next-item prediction samples.

  Parameters
  ----------
  histories : Dict[Any, List[int]]
      Mapping from user id to a list of interacted item ids (sorted by time).
  labels : Dict[Any, List[int]]
      Mapping from user id to a list of target item ids.
      Used only to filter users in eval mode (`uid in labels`).
  is_train : bool
      If True, generate multiple (prefix, next_item) samples per user.
      If False, return one sample per user (filtered by presence in `labels`).
  max_seq_len : int, default 100
      Maximum number of most recent items to keep in the returned history.

  Returns
  -------
  Dict[str, Any]
      Train mode (`is_train=True`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
            "label": int,
          }

      Eval mode (`is_train=False`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
          }

      where:
        - history["item_id"] contains up to `max_seq_len` last items of the selected prefix/history
        - history["length"] is the length of the returned (possibly truncated) history
        - label is a single next item id (int)

  Examples
  --------
  Train mode:
  >>> ds = YambdaDataset(histories, labels={}, is_train=True, max_seq_len=100)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  >>> s["label"]

  Eval mode (filters users by `labels` keys):
  >>> ds = YambdaDataset(histories, labels=test_targets, is_train=False)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  """

  def __init__(self,
      histories: Dict[Any, List[int]],
      labels: Dict[Any, List[int]],
      is_train: bool,
      max_seq_len: int = 100,
  ) -> None:
      super().__init__()
      self.histories = histories
      self.labels = labels
      self.is_train = is_train
      self.max_seq_len = max_seq_len

      self.samples: List[Tuple[Any, int]] = []

      if self.is_train:
        for uid, hist in self.histories.items():
            for t in range(1, len(hist)):
                self.samples.append((uid, t))

      else:
        for uid in self.labels.keys():
            if uid in self.histories:
                last_idx = len(self.histories[uid])
                self.samples.append((uid, last_idx))


  def __len__(self) -> int:
      """Return number of samples (prefix samples in train mode, users in eval mode)."""
      return len(self.samples)

  def __getitem__(self, idx: int) -> Dict[str, Any]:

    """
      Build and return a single sample using an index pointer (uid, t).

      In train mode: returns a truncated prefix and the next item as an integer label.
      In eval mode: returns the truncated full history.
    """
    res = {}
    if self.is_train:
        uid, t = self.samples[idx]
        hist = self.histories[uid]
        pref_hist = hist[:t][-self.max_seq_len:]
        res["uid"] = uid
        res["history"] = {
            "item_id": pref_hist,
            "length": len(pref_hist)
        }
        res["label"] = hist[t]

    else:
        uid, t = self.samples[idx]
        label = self.labels[uid]

        hist = self.histories[uid]
        pref_hist = hist[:t][-self.max_seq_len:]
        res["uid"] = uid
        res["history"] = {
            "item_id": pref_hist,
            "length": len(pref_hist)
        }

    return res

In [12]:
tests.test_yambda_dataset(YambdaDataset)

All good! :)


### Функция `collate_fn`

Реализуйте функцию `collate_fn`, которая будет использоваться в `DataLoader` для преобразования списка семплов
из `YambdaDataset` в батчи, удобные для подачи в модель и работы с ними.

Как говорилось ранее, мы используем flatten-представление: вместо padding до общей длины мы
1) конкатенируем все пользовательские истории в батче в один 1D-тензор  
2) отдельно сохраняем `length`, чтобы позже восстановить границы последовательностей


#### Вход

`batch: List[Dict[str, Any]]` — список семплов из `YambdaDataset`.

#### Что должна сделать `collate_fn`

Функция должна сформировать единый словарь, где все значения — `torch.Tensor` типа `torch.long`.

- `result["history"]["item_id"]` 1D тензор, полученный конкатенацией всех `history["item_id"]` в порядке объектов в `batch` размерность: `(sum(history_lengths),)`

- `result["history"]["length"]` 1D тензор длин историй для каждого объекта батча размерность: `(batch_size,)`

- `result["uid"]` 1D тензор идентификаторов пользователей размерность: `(batch_size,)`

- `result["label"]` (только если во входных семплах есть `"label"`) 1D тензор лейблов (next item id) в порядке объектов батча размерность: `(batch_size,)`

#### Требования

- Не использовать `padding`. Только `flatten`-конкатенация + `lengths`.
- Сохранять порядок объектов в `batch` при конкатенации.
- Возвращать `"label"` только если он присутствует во входных семплах.
- Все числовые значения должны быть приведены к `torch.Tensor` типа `torch.long`.


In [19]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
  """
  Collate function that converts a list of samples into a **flatten** batch representation.

  This function implements the "flatten" batching scheme: instead of padding variable-length
  sequences to a common length, it concatenates all user histories in the batch into a single
  1D tensor and returns a companion `length` tensor to recover per-user boundaries later.

  The function is compatible with `YambdaDataset` in two modes:
    - Train mode samples contain keys: `"uid"`, `"history"`, and `"label"` (where `"label"` is an `int`).
    - Eval mode samples contain keys: `"uid"` and `"history"`.

  Output batch format
  -------------------
  The returned dictionary contains:
    - `result["history"]["item_id"]`: 1D tensor with all history items concatenated in the
      order of samples in `batch`, shape `(sum(history_lengths),)`, dtype `torch.long`.
    - `result["history"]["length"]`: 1D tensor of per-sample history lengths,
      shape `(batch_size,)`, dtype `torch.long`.
    - `result["uid"]`: 1D tensor of user ids, shape `(batch_size,)`, dtype `torch.long`.
    - If `"label"` is present in the input samples (train batches):
        - `result["label"]`: 1D tensor of labels (next item ids), shape `(batch_size,)`,
          dtype `torch.long`.

  Parameters
  ----------
  batch : List[Dict[str, Any]]
      List of samples returned by the dataset `__getitem__`.

  Returns
  -------
  Dict[str, Any]
      A nested dictionary where all returned values are `torch.Tensor` objects.

  Examples
  --------
  - Train-mode: returns `"history"` + `"uid"` + `"label"` (1D tensor of next-item ids).
  - Eval-mode: returns `"history"` + `"uid"` (no `"label"` key).
  """

  '''
  {
    "uid":     tensor([uid0, uid1, uid2]),           # (batch_size,)
    "history": {
        "item_id": tensor([10,20,30, 5,6, 1,2,3,4]), # (sum_of_lengths,) — все склеены
        "length":  tensor([3, 2, 4]),                 # (batch_size,)
    },
    "label":   tensor([40, 7, 5]),                    # (batch_size,) — только если train
}

'''

  res = {}
  uids = torch.tensor([sample["uid"] for sample in batch], dtype = torch.long)
  history_items = [sample["history"]["item_id"] for sample in batch]

  history_items_paddle = []
  for h in history_items:
    history_items_paddle += h

  history_items_paddle = torch.tensor(history_items_paddle, dtype = torch.long)

  history_len = torch.tensor([sample["history"]["length"] for sample in batch], dtype = torch.long)
  res["uid"] = uids
  res["history"] = {"item_id":history_items_paddle, "length":history_len }
  if "label" in batch[0]:
     res["label"] = torch.tensor([sample["label"] for sample in batch], dtype=torch.long)

  return res


In [14]:
tests.test_collate_fn(collate_fn)

All good! :)


# 2. Реализация графа вычислений для обучения двухбашенной модели (без лосса) и инференса (получения кандидатов) (2 балла)

## UserEncoder

Реализуйте класс `UserEncoder`, который является основным компонентом нашей модели.

`UserEncoder` — это модуль, который по истории взаимодействий пользователя строит его контекстное представление.

Каждый пользователь $u$ описывается историей его взаимодействий $S_u$.

Каждому айтему $i$ из каталога соответствует обучаемый эмбеддинг $e_i \in \mathbb{R}^d$.

Представление для пользователя $u$, $P_u$, получается как агрегат эмбеддингов всех его предыдущих взаимодействий. В этом домашнем задании в качестве агрегации необходимо реализовать представление пользователя в духе
bag-of-words по его истории взаимодействий.

Для пользователя $u$ с историей взаимодействий $i_1, i_2, \ldots, i_{|S_u|}$ требуется получить:

$$
P_u = \sum_{k=1}^{|S_u|} e_{i_k}.
$$

#### Вход модели

Во время обучения данные из `YambdaDataset` с помощью `collate_fn` преобразуются в `flatten`-батчи и `batch["history"]` подается на вход метода `UserEncoder.forward` для получение представлений пользователей из батча.

#### Что должна сделать модель

1. Преобразовать полученные `item_id` в эмбеддинги объектов;
2. Посчитать представления пользователей в виде тензора размера `(batch_size, embedding_dim)`.
3. Вернуть полученные представления

In [20]:
class UserEncoder(nn.Module):
  """
  User encoder that represents each user by a cumulative prefix sum of item embeddings.

  Parameters
  ----------
  num_items : int
      Total number of unique items in the catalog.
      Item ids must be in ``[0, num_items - 1]``.
  embedding_dim : int
      Dimension of item embeddings.

  Forward input
  -------------
  inputs : Dict[str, torch.Tensor]
      Dictionary with keys:
      - "item_id": Flattened item indices for concatenated sequences,
        shape ``(total_num_events,)``, dtype ``torch.long``.
      - "length": Per-user sequence lengths, shape ``(batch_size,)``,
        dtype ``torch.long``.

  Forward output
  --------------
  torch.Tensor
      User representations, one vector per user, shape ``(batch_size, embedding_dim)``.
  """
  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.item_embeddings = nn.Embedding(num_items, embedding_dim)



  def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:

    item_ids = inputs["item_id"]
    length = inputs["length"] #(num_items)

    embeddings = self.item_embeddings(item_ids) # num_items_ids, embedding_dim || Матрица E - вызываем .weight
    # разъединям по length

    item_embedd_separated = torch.split(embeddings, length.tolist()) # (len(lengths) - кол во тензоров каждый имеет: (lengths_i, dim_size))

    user_emdeds_like_sum_items = [items_on_user.sum(dim=0) for items_on_user in item_embedd_separated]
    concat_all_users_emdeds_like_sum_items = torch.stack((user_emdeds_like_sum_items))

    return concat_all_users_emdeds_like_sum_items


In [16]:
layer = nn.Embedding(3,4)
print(layer.weight)



Parameter containing:
tensor([[-1.0944, -1.0305, -1.0846, -0.3041],
        [-0.9130, -0.9086,  0.7394,  1.6643],
        [ 0.2927,  0.1558, -0.4931, -1.1367]], requires_grad=True)


In [17]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TwoTowerModel: обучение и инференс

`TwoTowerModel` объединяет `UserEncoder` и логику обучения/инференса модели.

#### Обозначения

$\mathbf{E} \in \mathbb{R}^{|I| \times d}$ — таблица эмбеддингов айтемов

$\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя $u$

Релевантность айтема $i$ для пользователя $u$: $r_i = \langle \mathbf{E}_{i}, \mathbf{P}_{u}\rangle$.


#### Что должна делать модель

Нам дан батч:
`inputs["history"]`: история (то, на основе чего строим пользователя и обучаетмся)
`inputs["labels"]`: таргеты/позитивы (айтемы с последней недели, по которым хотим получать метрики на эвале)

Для каждого пользователя в батче:
- строим $\mathbf{U}$ по его истории

#### Режим обучения (`self.training == True`)

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- вычислить лосс через метод `compute_loss`
- вернуть лосс

#### Режим эвала (`self.training == False`):

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- посчитать: $\text{all\_scores} = \langle\mathbf{U}, \mathbf{E}^{\top}\rangle$ размера `(batch_size, num_items)`
- вернуть тензор `all_scores` (метрики считаются отдельно)


#### Откуда берутся позитивы на обучении

Обучение формулируется как задача `next item prediction`. Для каждого шага в пользовательской истории позитивным примером считается следующий айтем в последовательности пользователя. Иными словами, модель обучается предсказывать следующий объект взаимодействия на основе всех предыдущих.
    
    

In [12]:
class TwoTower(nn.Module):

  """
  Recommendation model combining user encoder with training and inference logic.

  The model produces:
    - a user representation vector `P_u` via `UserEncoder`
    - an item representation matrix `E` from the embedding table
    - uses dot-product relevance scores: `r_{ui} = <P_u, E_i>`.

  The `forward` method behaves differently depending on `self.training`:

  Training mode (`self.training == True`)
    - Encodes users.
    - Delegates loss computation to `compute_loss(...)`.
    - Returns a loss tensor.

  Evaluation / inference mode (`self.training == False`)
    - Encodes users.
    - Computes scores against all items in the catalog.
    - Returns a full score matrix.

  Parameters
  ----------
  num_items : int
    Total number of unique items in the catalog. Item ids must be in `[0, num_items - 1]`.
  embedding_dim : int
    Dimension of user/item embeddings.

  Notes
  -----
  This base class does not implement `compute_loss`. Subclasses should override it to define a training objective.
  """

  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.init_weights(0.02)

  @torch.no_grad()
  def init_weights(self, initializer_range: float) -> None:
    """
    Initialize all model parameters with truncated normal distribution.

    Parameters
    ----------
    initializer_range : float
        Standard deviation of the truncated normal initializer.
    """
    for key, value in self.named_parameters():
      assert "weight" in key
      nn.init.trunc_normal_(
        value.data,
        std=initializer_range,
        a=-2 * initializer_range,
        b=2 * initializer_range,
      )

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    """
    Compute training loss.

    Parameters
    ----------
    user_repr : torch.Tensor
        User representations returned by the encoder, shape ``(batch_size, embedding_dim)``.
    inputs : Dict[str, Any]
        Full input batch. Expected to contain at least:
          - ``inputs["history"]``: dict with flattened history fields
          - label information (e.g., ``inputs["label"]``), depending on the training setup

    Returns
    -------
    torch.Tensor
        Scalar loss tensor.
    """
    # Эту функцию мы реализуем отдельно позже!
    # Не трогать ее и не менять здесь!
    raise NotImplementedError

  def forward(self, inputs: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    """
    Run a forward pass with mode-dependent behavior.
    During training: computes and returns loss.
    During evaluation: computes and returns ranking scores for all items.

    Parameters
    ----------
    inputs : Dict[str, Any]
        Batch dictionary produced by `collate_fn`. Expected keys:
          - ``"history"``: dict with
                - ``"item_id"``: 1D flattened history item ids
                - ``"length"``: per-user history lengths
          - ``"uid"``: user ids tensor

    Returns
    -------
    torch.Tensor
        - If training (self.training == True): loss, scalar tensor
        - If evaluating (self.training == False): all_scores, relevance scores for all items with shape (batch_size, num_items)
    """

    user_as_sum_embedds = self.encoder(inputs["history"]) # U (batch, dim)
    if self.training:
      loss = self.compute_loss(user_repr=user_as_sum_embedds, inputs=inputs)
      return loss
    else:
      items_embeddings = self.encoder.item_embeddings.weight # E (|I|, dim)
      score = user_as_sum_embedds @ items_embeddings.T
      return score




In [19]:
tests.test_two_tower(TwoTower)

All good! :)


Посмотрим на то, что у нас получилось

In [21]:
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 2048


catalog_size = data["item_id"].n_unique()

train_histories = dict(
    train_df.group_by("uid")
    .agg(pl.col("item_id").sort_by("timestamp"))
    .iter_rows()
)

test_targets = dict(
    test_df.group_by("uid")
    .agg(pl.col("item_id"))
    .iter_rows()
)


yambda_train_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=True
)

yambda_eval_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=False
)

yambda_train_dataloader = DataLoader(
  dataset=yambda_train_dataset,
  batch_size=TRAIN_BATCH_SIZE,
  shuffle=True,
  collate_fn=collate_fn,
  drop_last=True
)

yambda_eval_dataloader = DataLoader(
  dataset=yambda_eval_dataset,
  batch_size=EVAL_BATCH_SIZE,
  shuffle=False,
  collate_fn=collate_fn,
  drop_last=False
)

# 3. Цикл обучения (1 балл)

Реализуйте функцию `evaluation`, которая выполняет оценку качества модели рекомендаций.

Функция должна:
1) Получить top-k рекомендаций для каждого пользователя из `dataloader`.
2) Собрать их в словарь формата `Dict[uid, List[item_id]]`.
3) Посчитать метрики, вызвав `evaluate(...)`, и вернуть результат.

#### Tips & Tricks
- Не забудьне перевести модель в `.eval()` режим
- Метрики можно считать с помощью функции `evaluate` из ДЗ 1

In [25]:
@torch.no_grad()
def evaluation(
  dataloader: DataLoader, # хранятся батчи
  model: TwoTower,
  catalog_size: int,
  topk: int,
  device: str = "cuda",
) -> Tuple[torch.Tensor, torch.Tensor]:

  model.eval()
  recomendations = {}


  for batch in dataloader:
    batch["uid"] = batch["uid"].to(device)
    batch["history"]["item_id"] = batch["history"]["item_id"].to(device)
    batch["history"]["length"] = batch["history"]["length"].to(device)

    uids = batch["uid"]

    scores = model(batch)  # twnsor(U, D) tensor[ [], [], [], ...]
        # формируем topk по скорам - каждый скор это схожесть
    topk_res = torch.topk(scores, k=topk) # tensor(U, topk)

    for uid, items in zip(uids, topk_res.indices):
      recomendations[uid.item()] = items.cpu().tolist()

    targets = dataloader.dataset.labels

  return evaluate(targets,recomendations,catalog_size,topk)



In [22]:
# a = [1,2]
# b = torch.tensor([[1,2,3],[10,12,13]])

# for x,y in zip(a,b):
#     print(x,y)

In [23]:
# x = torch.arange(1., 6.)
# print(x)
# torch.topk(x,2)

Реализуйте функцию `train`, которая обучает модель и после каждой эпохи запускает валидацию.

После завершения каждый эпохи необходимо:
- Запустить функцию `evaluation` на `valid_dataloader`
- Вывести метрики валидации в читаемом виде.
- Посчитать и вывести средний лосс за эпоху.

После окончания обучения вывести сообщение о завершении и вернуть состояние модели (`state dict`).

#### Примечания

- Важно корректно переключать режимы модели:
  - обучение выполняется в `train` режиме,
  - валидация должна выполняться внутри `evaluation`, где модель переводится в `eval` режим.
- Перенос батча на `device` должен корректно работать со структурой батча, где могут встречаться вложенные словари с тензорами.

In [23]:
def train(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    catalog_size: int,
    topk: int,
    device: str = "cuda"
  ) -> torch.nn.Module:

  avg_loss_per_epoch = []
  metrics_per_epoch = []

  for epoch in range(num_epochs):

    model.train()
    total_loss = 0
    num_batches = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch}"):
      num_batches +=1
      batch["uid"] = batch["uid"].to(device)
      batch["history"]["item_id"] = batch["history"]["item_id"].to(device)
      batch["history"]["length"] = batch["history"]["length"].to(device)
      batch["label"] = batch["label"].to(device)

      loss = model(batch) # -> loss
      loss.backward()
      optimizer.step()

      optimizer.zero_grad()
      total_loss += loss.item()

    avg_loss = total_loss/num_batches
    avg_loss_per_epoch.append(avg_loss)


    metrics = evaluation(valid_dataloader,model, catalog_size, topk, device)
    print(metrics)
    metrics_per_epoch.append(metrics)

    model.train()

  return model.state_dict()


In [25]:
# epochs = range(num_epochs)

# plt.plot(epochs, avg_loss_per_epoch)
# plt.title("loss per epoch")

# ndcgs = [m["ndcg"] for m in metrics_per_epoch]
# plt.plot(epochs, ndcgs)
# plt.title("nDCG per epoch")


# Реализуем различные способы обучения полученной двухбашенной модели

In [16]:
NUM_EPOCHS = 1
LEARNING_RATE = 1e-3
DEVICE = "cuda"

## 4. Softmax loss (1 балл)



В задаче отбора кандидатов каждому пользователю и каждому айтему сопоставляется векторное представление размерности $d$ в общем латентном пространстве.

Скор релевантности пользователя $u$ и айтема $i$ вычисляется как скалярное произведение их эмбеддингов:

$$
r(u, i) = \langle \mathbf{E}_i,\;\mathbf{P}_u \rangle,
$$

где:
- $\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя, полученное из `UserEncoder`;
- $\mathbf{E}_i \in \mathbb{R}^d$ — обучаемое представление айтема.

Чем больше значение $r(u, i)$, тем более релевантным считается айтем $i$ для пользователя $u$.

На этапе инференса айтемы ранжируются по убыванию релевантности, и модель возвращает top-K кандидатов.

В этом задании мы обучаем модель на задачу экстремальной многоклассовой классификации.

Для каждого пользователя в батче нужно предсказать один правильный айтем из всего каталога айтемов размера $|\mathcal{I}|$.

#### Формула

Пусть $i^+$ — следующий айтем для пользователя $u$, тогда:

$$
\mathcal{L}_{\text{softmax}} = - \sum_{u \in \mathbf{U}} \log p(i^+ \mid u) = - \sum_{u \in \mathbf{U}} \left[r(u, i^+) - \log \sum_{j\in\mathcal{I}} \exp(r(u,j))\right].
$$


#### Почему это лучший способ обучать модели для этой стадии

Full softmax использует информацию обо всём каталоге: обучение модели эквивалентно применению: мы ищем позитив из всего каталога на обучении, мы берем top-K айтемов из всего каталога на применении.

In [27]:
print("catalog_size:", catalog_size)
print("max item_id:", data["item_id"].max())
print("n_unique:", data["item_id"].n_unique())

catalog_size: 157357
max item_id: 157356
n_unique: 157357


In [28]:
class SoftmaxModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    loss = 0
    E = self.encoder.item_embeddings.weight
    positives = inputs["label"]
    scores = user_repr @ E.T
    loss = F.cross_entropy(scores, positives)

    return loss

In [29]:
tests.test_softmax_model(SoftmaxModel)

All good! :)


In [30]:
gc.collect()
torch.cuda.empty_cache()


model_full = SoftmaxModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_full = torch.optim.Adam(params=model_full.parameters(), lr=LEARNING_RATE)
best_checkpoint_full  = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_full,
    optimizer=optimizer_full,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [10:22<00:00,  6.02it/s]


{'hitrate': 0.3263632964802649, 'recall': 0.10409070143870766, 'ndcg': np.float64(0.03780557010167118), 'coverage': 0.4492777569475778}


In [31]:
# print("catalog_size:", catalog_size)
# print("max item in train:", train_df["item_id"].max())
# print("max item in test:", test_df["item_id"].max())

In [32]:
model_full.load_state_dict(best_checkpoint_full)
final_metrics_full = evaluation(
    yambda_eval_dataloader,
    model_full,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_recs(final_metrics_full)

All good! :)


In [33]:
print(final_metrics_full)

{'hitrate': 0.3263632964802649, 'recall': 0.10409070143870766, 'ndcg': np.float64(0.03780557010167118), 'coverage': 0.4492777569475778}


## 5. BCE loss (1 балл)

Главная проблема предыдущего подхода — вычисления и память: полный softmax обычно применим, когда каталог не слишком большой — примерно до десятков/сотен тысяч айтемов. Для каталогов в миллионы обычно используют более простые подходы. Пойдет по их усложнению. Самый простой из них Binary Cross-Entropy (BCE).

Для каждого пользователя $u$ мы рассматриваем:

- позитивный пример: айтем $i^+$, с которым пользователь действительно взаимодействовал (следующий после истории пользователя);
- негативные примеры: айтемы $i^-$, сэмплированные из каталога (обычно равномерно), с которыми пользователь не взаимодействовал.

Модель обучается предсказывать вероятность того, что айтем является релевантным для пользователя в данный момент времени.

#### Формула

Для одного пользователя $u$, позитивного айтема $i^+$ и множества негативных айтемов $\mathcal{I}^-$ функция потерь имеет вид:

$$
\mathcal{L}_{\text{BCE}} =
- \Big[
\log \sigma\bigl(r(u, i^+)\bigr)
+ \sum_{i^- \in \mathcal{I}^-}
\log \bigl(1 - \sigma(r(u, i^-))\bigr)
\Big]
$$

In [34]:
class BCEModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    E = self.encoder.item_embeddings.weight # каталог
    num_items = E.shape[0]
    num_users = user_repr.shape[0]
    cnt_negatives_per_user = 2048


    Loss_positive_path = F.logsigmoid((user_repr * E[inputs["label"]]).sum(dim = -1)) # <U_i, E_i> -> вектор -> sum -> score (типо как матрица скоров)
    items_negatives_indx = torch.randint(0, num_items, (num_users, cnt_negatives_per_user), device=user_repr.device)
    neg_sim = (user_repr.unsqueeze(1) * E[items_negatives_indx]).sum(dim = -1)
    Loss_negative_path = F.logsigmoid(-neg_sim)



    return -(Loss_positive_path.mean() + Loss_negative_path.mean())

In [35]:
gc.collect()
torch.cuda.empty_cache()

model_bce = BCEModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bce = torch.optim.Adam(params=model_bce.parameters(), lr=LEARNING_RATE)
best_checkpoint_bce = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bce,
    optimizer=optimizer_bce,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [08:33<00:00,  7.30it/s]


{'hitrate': 0.19550819847246703, 'recall': 0.04996345764348274, 'ndcg': np.float64(0.01744913530999845), 'coverage': 0.20859574089490776}


In [36]:
model_bce.load_state_dict(best_checkpoint_bce)
final_metrics_bce = evaluation(
    yambda_eval_dataloader,
    model_bce,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bce_recs(final_metrics_bce)

All good! :)


In [37]:
print(final_metrics_bce)

{'hitrate': 0.19550819847246703, 'recall': 0.04996345764348274, 'ndcg': np.float64(0.01744913530999845), 'coverage': 0.20859574089490776}


## 6. BPR loss (1 балл)

Помимо BCE, двухбашенные модели также могут обучаться с использованием Bayesian Personalized Ranking (BPR).

Модель обучается на парах айтемов:

- позитивный айтем $i^+$, с которым пользователь действительно взаимодействовал;
- негативный айтем $i^-$, с которым пользователь не взаимодействовал (в данном подходе выбирается равномерно из каталога).

Цель обучения — добиться, чтобы для каждого пользователя выполнялось:

$$
r(u, i^+) > r(u, i^-)
$$

Таким образом, BPR напрямую приближает оптимизацию метрик ранжирования (Recall@K, nDCG@K), что делает его подходящим для retrieval-моделей.

#### Формула

Для каждого пользователя $u$ выбирается один позитивный айтем $i^+$ и один негативный айтем $i^-$. Функция потерь BPR определяется как:

$$
\mathcal{L}_{\text{BPR}}
= - \sum_{u \in \mathbf{U}} \log \sigma \bigl(r(u, i^+) - r(u, i^-)\bigr),
$$

где:
- $\mathbf{U}$ - набор польователей в батче;
- $r(u, i)$ — скор релевантности пользователя $u$ и айтема $i$;
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида.

In [38]:
class BPRModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:

    E = self.encoder.item_embeddings.weight # каталог
    num_items = E.shape[0]
    num_users = user_repr.shape[0]

    positive_items_indx = inputs["label"]
    positive_items = E[positive_items_indx]
    positive_scores = (user_repr * positive_items).sum(dim = -1)

    negative_items_indx = torch.randint(0,num_items, (num_users,), device = user_repr.device) # один негативынй айтем на юзера
    negative_items = E[negative_items_indx]
    negative_scores = (user_repr * negative_items).sum(dim = -1)

    Loss = -F.logsigmoid(positive_scores - negative_scores).mean()
    return Loss

In [39]:
gc.collect()
torch.cuda.empty_cache()

model_bpr = BPRModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bpr = torch.optim.Adam(params=model_bpr.parameters(), lr=LEARNING_RATE)
best_checkpoint_bpr = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bpr,
    optimizer=optimizer_bpr,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [06:21<00:00,  9.83it/s]


{'hitrate': 0.22982428029696095, 'recall': 0.06297379962337925, 'ndcg': np.float64(0.021767383627489978), 'coverage': 0.14402918205100504}


In [40]:
model_bpr.load_state_dict(best_checkpoint_bpr)
final_metrics_bpr = evaluation(
    yambda_eval_dataloader,
    model_bpr,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bpr_recs(final_metrics_bpr)

All good! :)


## 7. Sampled softmax, uniform negatives (1 балл)

Sampled softmax — это аппроксимация полного softmax: вместо всех айтемов мы берём небольшой их набор и считаем softmax только по нему.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$.

#### Формула

$$
\mathcal{L}_{\text{sampled-uniform}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [54]:
class SampledUniformModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)


  def compute_loss(self, user_repr, inputs):
      E = self.encoder.item_embeddings.weight
      num_items = E.shape[0]
      num_users = user_repr.shape[0]

      r_pos = (user_repr * E[inputs["label"]]).sum(dim=-1, keepdim=True)   # (B, 1)
      neg_ids = torch.randint(0, num_items, (num_users, self.num_negatives), device=user_repr.device)
      E_neg = E[neg_ids]                                                    # (B, N, d)
      r_neg = (user_repr.unsqueeze(1) * E_neg).sum(dim=-1)                 # (B, N)
      logits = torch.cat([r_pos, r_neg], dim=1)                            # (B, 1+N)
      target = torch.zeros(num_users, dtype=torch.long, device=user_repr.device)
      return F.cross_entropy(logits, target)

In [55]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_uniform = SampledUniformModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_uniform = torch.optim.Adam(params=model_sampled_uniform.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_uniform = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_uniform,
    optimizer=optimizer_sampled_uniform,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [08:40<00:00,  7.20it/s]


{'hitrate': 0.328072424290979, 'recall': 0.10464740736648304, 'ndcg': np.float64(0.03797564242778629), 'coverage': 0.43554465324072017}


In [56]:
model_sampled_uniform.load_state_dict(best_checkpoint_sampled_uniform)
final_metrics_sampled_uniform = evaluation(
    yambda_eval_dataloader,
    model_sampled_uniform,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_uniform_recs(final_metrics_sampled_uniform)

All good! :)


In [46]:
print(final_metrics_sampled_uniform)

{'hitrate': 0.2904983175773113, 'recall': 0.08686282700430044, 'ndcg': np.float64(0.03075602411417891), 'coverage': 0.36729220816360253}


In [ ]:
a = torch.tensor([1,2,3])
a.sum(dim = -1)

## 8. Sampled softmax, in-batch negatives (1 балл)

В прошлой задаче мы приближали полный softmax, сэмплируя негативы равновероятно из каталога. Теперь рассмотрим ещё более популярный подход: использование in-batch негативов.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$ (только теперь мы семплируем не из всего каталога, а из батча).

#### Формула

$$
\mathcal{L}_{\text{sampled-batch}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

```
label = [12, 33, 8, 77]
Для user_0 (позитив=12):  негативы = [33, 8, 77]  ← label-ы остальных
Для user_0 (позитив=33):  негативы = [12, 8, 77]
Для user_1 (позитив=8):   негативы = [12, 33, 77]
Для user_2 (позитив=77):  негативы = [12, 33, 8]
```

In [61]:
class SampledInBatchModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)

  # def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
  #   E = self.encoder.item_embeddings.weight # каталог
  #   num_items = E.shape[0]
  #   dim_items = E.shape[1]
  #   num_users = user_repr.shape[0]

  #   all_item = E[inputs["label"]]
  #   all_scores = user_repr @ all_item.T

  #   target = torch.arange(num_users, device=user_repr.device)
  #   Loss = F.cross_entropy(all_scores, target)


  #   return Loss
  def compute_loss(self, user_repr, inputs):
    E = self.encoder.item_embeddings.weight
    num_items = E.shape[0]
    num_users = user_repr.shape[0]

    all_items = E[inputs["label"]]
    scores_batch = user_repr @ all_items.T
    # + Uniform негативы (B, N)
    neg_ids = torch.randint(0, num_items, (self.num_negatives,), device=user_repr.device)
    scores_uniform = user_repr @ E[neg_ids].T
    # вместе
    logits = torch.cat([scores_batch, scores_uniform], dim=1)   # (B, B+N)
    target = torch.arange(num_users, device=user_repr.device)   # позитив на диагонали
    return F.cross_entropy(logits, target)

In [62]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch = SampledInBatchModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_in_batch = torch.optim.Adam(params=model_sampled_in_batch.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch,
    optimizer=optimizer_sampled_in_batch,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [06:28<00:00,  9.64it/s]


{'hitrate': 0.294797842226139, 'recall': 0.08896167538677287, 'ndcg': np.float64(0.03294786698633812), 'coverage': 0.5073431750732411}


In [63]:
model_sampled_in_batch.load_state_dict(best_checkpoint_sampled_in_batch)
final_metrics_sampled_in_batch = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_recs(final_metrics_sampled_in_batch)

All good! :)


In [64]:
final_metrics_sampled_in_batch

{'hitrate': 0.294797842226139,
 'recall': 0.08896167538677287,
 'ndcg': np.float64(0.03294786698633812),
 'coverage': 0.5073431750732411}

## 9. Sampled softmax, in-batch negatives + logq correction (1 балл)

Использование in-batch подход это быстро и эффективно, но остаётся важная проблема: такое распределение негативов не совпадает с распределением в случае полного или uniform sampled softmax.

Один из стандартных способов избавиться от смещения — добавить log-q коррекцию.

#### Почему нужна коррекция

Обычный in-batch подход воспринимает все негативы как “равноправные”, но в реальности некоторые айтемы встречаются гораздо чаще, другие — почти никогда.

То есть негативы получаются как выборка из некоторого распределения $q(i)$, а не равномерные. Если мы хотим приблизиться к полному softmax, нужно компенсировать это смещение.

#### Формула

Пусть $q(i)$ — вероятность того, что айтем $i$ будет появляться как негатив-кандидат.

Тогда корректируем логит негативов:

$$
\tilde{r}(u,i) = r(u,i) - \log q(i),
$$

где $q(i)$ — вероятность появления айтема $i$ в качетстве негатива: частота его появления среди всех позитивов: $\frac{\#i}{\#all}$.

In [10]:
def build_q_from_train_interactions(
  train_data: pl.DataFrame,
  catalog_size: int,
  item_col: str = "item_id",
  eps: float = 1e-12,
) -> torch.Tensor:


  cnt = torch.zeros(catalog_size)

  for el in train_data[item_col]:
    cnt[el] += 1

  q = cnt/ len(train_data[item_col])
  q = q + eps

  return q


In [14]:
class SampledInBatchModelLogQ(TwoTower):
  def __init__(
    self,
    num_items: int,
    embedding_dim: int,
    num_negatives: int,
    q: torch.Tensor,
    eps: float = 1e-12,
  ) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.eps = eps

    q = q.detach().float()
    q = q / (q.sum() + eps)
    logq = torch.log(q.clamp_min(eps))
    self.register_buffer("logq", logq)

    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    num_users = user_repr.shape[0]
    E = self.encoder.item_embeddings.weight
    all_items = E[inputs["label"]]
    scores_batch = user_repr @ all_items.T

    scores_batch = scores_batch - self.logq[inputs["label"]].unsqueeze(0) # чтобы броадкастинг сработал

    target = torch.arange(num_users, device=user_repr.device)
    return F.cross_entropy(scores_batch, target)




In [26]:
gc.collect()
torch.cuda.empty_cache()

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
q = build_q_from_train_interactions(train_df, catalog_size)
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

model_sampled_in_batch_logq = SampledInBatchModelLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq = torch.optim.Adam(params=model_sampled_in_batch_logq.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq,
    optimizer=optimizer_sampled_in_batch_logq,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

Epoch 0: 100%|██████████| 3747/3747 [06:44<00:00,  9.27it/s]


{'hitrate': 0.3352828072424291, 'recall': 0.10829775909333589, 'ndcg': np.float64(0.039251459065041704), 'coverage': 0.43568446271853173}


In [27]:
model_sampled_in_batch_logq.load_state_dict(best_checkpoint_sampled_in_batch_logq)
final_metrics_sampled_in_batch_logq = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_recs(final_metrics_sampled_in_batch_logq)

All good! :)


# Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

In [28]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled softmax, uniform negatives", **final_metrics_sampled_uniform},
    {"method": "Sampled softmax, in-batch negatives", **final_metrics_sampled_in_batch},
    {"method": "Sampled softmax, in-batch negatives + logq correction", **final_metrics_sampled_in_batch_logq},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

NameError: name 'final_metrics_full' is not defined

## 10. Вопросы на понимание (1 балл)

1. В чем основная проблема использования `Full softmax`? в его знаменателе- мы должны пробежать по всему каталогу айтемов, что при большом размере очень долго или просто нереально. Нужно как-то аппроксимировать его.
2. Почему `BCE` хуже показал себя чем `BPR`?  BCE - poinwize способ обучения, это хуже чем pairwise. У нас в BCE для пары (u,d) предсказывается ответ пользователя: релевантен или нет, нет сравнения между айтемами. Негативы сэмплируются случайно из каталога. У BRP - он во первых pairwise подход к обучению. Мы показываем сетке позитив и негатив айтема для пользователя и учим сетку что бы скор для позитива был выше чем для негатива. один позитив и случайный негатив на юзера. глобальная пробелма в том, что он видит только одну пару, в то время как весь softmax весь каталог

3. В чем может быть проблема с `Sampled softmax, uniform` подходом? негативы слишком легкие для модели в таком случае. у нас мало хард негативов, которые нужны модели чтобы эффетивно обучаться.


4. В чем проблема `in-batch` подхода без использования `logq`-коррекции? in batch негативы приводят к смещению. Проблема в популярных айтемах. Так как у популряных атеймов много юзеров (много кто выбирал) то они часто будут находить в негативах. Модель видит корреляцию между популярным и негативным и начинает штрафовать за выдачу популярных айтемов, даже если они могут быть релевантными.
5. Почему при добавлении `logq` у нас упал `coverage`?

Ответы - текстом